# Ghi livestream camera đường phố thành video

Notebook chỉ **ghi hình** livestream camera (tạm thời: EarthCam Dublin, khu Temple Bar; camera demo dự kiến: 大阪 心斎橋 ユニクロ前) trong `DURATION_HOURS` giờ, làm dữ liệu cho notebook `app_shinsaibashi` (đếm người, ROI, cảnh báo) và để gán nhãn frame về sau. Không cần GPU.

- Luồng video được **sao chép nguyên vẹn** bằng ffmpeg (không mã hóa lại), nên CPU gần như rảnh và chất lượng giữ nguyên. Âm thanh bị bỏ.
- Video được chia thành **đoạn `SEGMENT_MINUTES` phút** (`.mkv`). Nếu bị ngắt giữa chừng, các đoạn đã ghi vẫn dùng được; mất kết nối thì notebook tự nối lại.
- `recording_info.json` ghi **thời điểm bắt đầu thật của từng đoạn**, để notebook ứng dụng hiển thị đúng giờ địa phương của camera.

**Chạy trên Kaggle**

1. *Settings:* Accelerator **None** (CPU), **Internet On**.
2. Bấm **Save Version → Save & Run All** để chạy nền. Đóng trình duyệt vẫn chạy tiếp, còn chạy tương tác thì sẽ bị dừng khi phiên hết hạn.
3. **Chọn giờ chạy** để ghi được lúc đông:

   | Camera | Giờ đông (địa phương) | Giờ Việt Nam |
   |---|---|---|
   | Dublin, Temple Bar (`Europe/Dublin`, mùa đông chậm hơn Việt Nam 7 tiếng) | khoảng 15–23h | khoảng 22h–6h sáng hôm sau |
   | Shinsaibashi (`Asia/Tokyo`, nhanh hơn Việt Nam 2 tiếng) | khoảng 18–22h | khoảng 16–20h |
4. Xong thì **Add Input** output của notebook này vào `app_shinsaibashi`, rồi đặt `SOURCE = 'video'`.

Dung lượng: 1080p khoảng 2–3 GB mỗi giờ; giới hạn `/kaggle/working` là 20 GB.

In [ ]:
import csv, glob, json, os, shutil, subprocess, sys, time

import cv2
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

# Camera. TẠM dùng EarthCam Dublin (khu Temple Bar, có người thật) trong khi chưa lấy được livestream Shinsaibashi.
# Đổi về Shinsaibashi: dán link đang phát của kênh FST.NET (大阪 心斎橋 ユニクロ前) và đặt TZ = 'Asia/Tokyo'.
YOUTUBE_URL = 'https://www.youtube.com/watch?v=3nyPER2kzqk'   # EarthCam Live: Dublin, Ireland
COOKIES_FILE = None        # nếu YouTube đòi xác minh "not a bot": cookies.txt (định dạng Netscape) trong /kaggle/input
DURATION_HOURS = 3.0
SEGMENT_MINUTES = 30
MAX_HEIGHT = 1080          # 720 nếu muốn nhẹ hơn (khoảng một nửa dung lượng), nhưng đầu người sẽ nhỏ hơn
TZ = 'Europe/Dublin'       # giờ địa phương của camera (Shinsaibashi: 'Asia/Tokyo')
OUT_DIR = '/kaggle/working/recording' if os.path.isdir('/kaggle/working') else 'recording'
os.makedirs(OUT_DIR, exist_ok=True)

# yt-dlp cần trình chạy JavaScript (deno) để giải mã chống tải tự động của YouTube
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'yt-dlp[default]', 'deno', 'imageio-ffmpeg'], check=True)
os.environ['PATH'] = os.path.dirname(sys.executable) + os.pathsep + os.environ.get('PATH', '')
FFMPEG = shutil.which('ffmpeg')
if FFMPEG is None:
    import imageio_ffmpeg
    FFMPEG = imageio_ffmpeg.get_ffmpeg_exe()
print('ffmpeg:', FFMPEG)
print('Ổ đĩa trống:', f"{shutil.disk_usage(OUT_DIR).free / 1e9:.1f} GB")

## 1. Lấy link luồng video

In [ ]:
def youtube_stream(url, verbose=False):
    """(link HLS, tiêu đề, độ cao, header HTTP) của luồng có hình cao nhất nhưng không quá MAX_HEIGHT."""
    import yt_dlp
    opts = {'quiet': not verbose, 'no_warnings': not verbose, 'ignore_no_formats_error': True}
    if COOKIES_FILE:
        opts['cookiefile'] = COOKIES_FILE
    with yt_dlp.YoutubeDL(opts) as ydl:
        info = ydl.extract_info(url, download=False, process=False)
    if info.get('_type') == 'url' and info.get('url') and info['url'] != url:
        return youtube_stream(info['url'], verbose)
    fmts = [f for f in info.get('formats') or [] if f.get('url') and f.get('vcodec') not in (None, 'none')]
    if not fmts:
        if not verbose:
            return youtube_stream(url, verbose=True)
        raise RuntimeError('YouTube không trả luồng video nào (xem cảnh báo của yt-dlp ở trên): thiếu deno, '
                           'YouTube chặn IP Kaggle (cần COOKIES_FILE), hoặc livestream không còn phát.')
    ok = [f for f in fmts if (f.get('height') or 0) <= MAX_HEIGHT] or fmts
    best = max(ok, key=lambda f: ((f.get('height') or 0), 'm3u8' in (f.get('protocol') or '')))
    return best['url'], info.get('title'), best.get('height'), best.get('http_headers') or {}


_, TITLE, HEIGHT, _ = youtube_stream(YOUTUBE_URL)
print(f'Livestream: {TITLE} | ghi ở {HEIGHT}p')

## 2. Ghi hình

Mỗi lần ffmpeg chạy (một "lượt") ghi tới khi đủ thời lượng hoặc mất kết nối. Mất kết nối thì lấy link mới và chạy lượt tiếp theo với thời gian còn lại. Cứ 10 phút in tiến độ một lần.

In [ ]:
def write_info(runs):
    """Gom danh sách đoạn của mọi lượt ghi, kèm thời điểm bắt đầu thật của từng đoạn."""
    segments = []
    for run in runs:
        if not os.path.exists(run['list']):
            continue
        with open(run['list'], newline='') as fh:
            rows = [r for r in csv.reader(fh) if len(r) >= 3]
        if not rows:
            continue
        t0 = float(rows[0][1])
        for name, start, end in (r[:3] for r in rows):
            segments.append({'file': name, 'start_epoch': run['start_epoch'] + float(start) - t0,
                             'duration_s': float(end) - float(start)})
    info = {'url': YOUTUBE_URL, 'title': TITLE, 'height': HEIGHT, 'timezone': TZ, 'runs': runs, 'segments': segments}
    with open(os.path.join(OUT_DIR, 'recording_info.json'), 'w', encoding='utf-8') as fh:
        json.dump(info, fh, ensure_ascii=False, indent=2)
    return segments


def folder_gb():
    return sum(os.path.getsize(p) for p in glob.glob(os.path.join(OUT_DIR, '*.mkv'))) / 1e9


start_all = time.time()
end_at = start_all + DURATION_HOURS * 3600
runs, failures = [], 0
print(f"Ghi từ {pd.Timestamp(start_all, unit='s', tz='UTC').tz_convert(TZ):%Y-%m-%d %H:%M} đến "
      f"{pd.Timestamp(end_at, unit='s', tz='UTC').tz_convert(TZ):%H:%M} giờ địa phương")
while time.time() < end_at - 30 and failures < 10:
    try:
        url, _, _, headers = youtube_stream(YOUTUBE_URL)
    except Exception as e:
        failures += 1
        print(f'  không lấy được link ({e}); thử lại sau 60 s')
        time.sleep(60)
        continue
    t_run = time.time()
    tag = str(int(t_run))
    remaining = end_at - t_run
    list_path = os.path.join(OUT_DIR, f'segments_{tag}.csv')
    cmd = [FFMPEG, '-hide_banner', '-loglevel', 'error', '-rw_timeout', '30000000']
    if headers:
        cmd += ['-headers', ''.join(f'{k}: {v}\r\n' for k, v in headers.items())]
    cmd += ['-i', url, '-t', f'{remaining:.0f}', '-map', '0:v:0', '-c', 'copy', '-an',
            '-f', 'segment', '-segment_time', str(SEGMENT_MINUTES * 60), '-reset_timestamps', '1',
            '-segment_list', list_path, '-segment_list_type', 'csv',
            os.path.join(OUT_DIR, f'part_{tag}_%03d.mkv')]
    proc = subprocess.Popen(cmd)
    last_report = time.time()
    while proc.poll() is None:
        if time.time() > end_at + 300:          # ffmpeg treo: dừng hẳn
            proc.terminate()
            break
        if time.time() - last_report >= 600:
            last_report = time.time()
            print(f'  {(time.time() - start_all) / 60:5.0f} phút | {folder_gb():.2f} GB')
        time.sleep(10)
    runs.append({'tag': tag, 'start_epoch': t_run, 'returncode': proc.returncode, 'list': list_path})
    write_info(runs)
    if time.time() < end_at - 30:
        failures += 1
        print(f'  ffmpeg dừng sau {(time.time() - t_run) / 60:.1f} phút (mã {proc.returncode}); nối lại...')
        time.sleep(5)

segments = write_info(runs)
print(f'Xong: {len(segments)} đoạn, {sum(s["duration_s"] for s in segments) / 3600:.2f} giờ video, '
      f'{folder_gb():.2f} GB, {failures} lần ngắt kết nối.')

## 3. Kiểm tra bản ghi

Danh sách đoạn và một frame ở giữa mỗi đoạn, kèm giờ địa phương.

In [ ]:
seg = pd.DataFrame(segments)
seg['bắt đầu (giờ địa phương)'] = pd.to_datetime(seg['start_epoch'], unit='s', utc=True).dt.tz_convert(TZ).dt.strftime('%Y-%m-%d %H:%M:%S')
seg['phút'] = (seg['duration_s'] / 60).round(1)
seg['MB'] = [os.path.getsize(os.path.join(OUT_DIR, f)) / 1e6 if os.path.exists(os.path.join(OUT_DIR, f)) else 0 for f in seg['file']]
display(seg[['file', 'bắt đầu (giờ địa phương)', 'phút', 'MB']].round(1))

n = len(seg)
fig, axes = plt.subplots(1, max(n, 1), figsize=(5.5 * max(n, 1), 3.6), squeeze=False)
for ax, (_, s) in zip(axes[0], seg.iterrows()):
    cap = cv2.VideoCapture(os.path.join(OUT_DIR, s['file']))
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    cap.set(cv2.CAP_PROP_POS_MSEC, s['duration_s'] / 2 * 1000)
    ok, frame = cap.read()
    cap.release()
    w_h = f"{frame.shape[1]}×{frame.shape[0]}, {fps:.0f} fps" if ok else 'không đọc được'
    mid = pd.Timestamp(s['start_epoch'] + s['duration_s'] / 2, unit='s', tz='UTC').tz_convert(TZ)
    if ok:
        ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    ax.set_title(f"{mid:%H:%M} giờ địa phương | {w_h}", fontsize=9); ax.axis('off')
plt.tight_layout(); plt.show()